In [1]:
#Pull the lesson pages straight from the course repository. 
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

### Q1. How many lesson pages

How many lesson pages are in the dataset?

24
72
240
720


In [2]:
len(files)

72

**Answer: 72**

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
from minsearch import Index

In [5]:
#Create a function to define the KB index
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [6]:
index = build_index(documents)

In [7]:
index

In [8]:
from google import genai
genai_client = genai.Client()

In [9]:
from rag_helper import RAGBase
assistant = RAGBase(index, genai_client)

## Q2. Indexing and searching

Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:
How does the agentic loop keep calling the model until it stops?

In [10]:
index.search('How does the agentic loop keep calling the model until it stops?', num_results=5)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

What's the filename of the first result?

01-agentic-rag/lessons/03-rag.md

01-agentic-rag/lessons/14-agentic-loop.md

04-evaluation/lessons/13-llm-as-judge.md

06-best-practices/lessons/02-hybrid-search.md



**Answer:** 01-agentic-rag/lessons/14-agentic-loop.md

In [11]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

('The agentic loop keeps calling the model until it stops by using a `while` loop with a specific exit condition.\n\nHere\'s how it works:\n\n1.  **Initial Call:** The loop starts by sending the user\'s question and initial instructions to the language model (LLM).\n2.  **Processing Response:** The LLM\'s response is processed.\n    *   **Function Call:** If the LLM decides it needs to use a tool (like `search`), its response will contain a `function_call` item.\n        *   A `has_function_calls` flag is set to `True`.\n        *   The specified function is executed (e.g., `make_call` for `search`).\n        *   The output of the function call is appended to the message history.\n    *   **Message:** If the LLM provides a message (an answer or an intermediate thought) without requesting a tool call, the `has_function_calls` flag remains `False` (or is reset to `False` at the beginning of each iteration).\n3.  **Loop Continuation:** The loop continues to the next iteration.\n    *   **

### Q3. RAG
Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

Two solutions are possible:

Implement the RAG flow yourself
Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
Build a RAG over the index from Q2 and answer the query:

How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

700

7000

70000

700000

We count input tokens instead of price because the cost depends on the model and provider you use, but the size of the prompt we send is the same for everyone.

Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). We'll read the input tokens from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, llm returns only the text. Modify it to return the whole response, and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).

Note: for this question and the next ones, if your answer doesn't match exactly, just select the closest option - especially if you use a different model or a different LLM provider.



**Answer:** token_count=7963

## Q4. Chunking
The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: chunk_documents. It uses a sliding window - a window of size characters slides across the text in steps of step characters, and each window position becomes one chunk:

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

With size=2000 and step=1000 (you can see the implementation here):

Each chunk is a window of size characters of the page.
The window moves forward by step characters between chunks. Since step is smaller than size, consecutive chunks overlap by size - step (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
Every chunk keeps the original fields (filename) and adds start (the offset in the page) and content (the chunk text).
How many chunks do you get?

70
295
1100
4500

In [13]:
len(chunks)

295

**Answer**: 295

## Q5. RAG with chunking
Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

about the same
3× fewer
10× fewer
30× fewer

In [14]:
index = build_index(chunks)

In [15]:
assistant = RAGBase(index, genai_client)

In [16]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

('The agentic loop keeps calling the model using a `while True` loop.\n\nHere\'s how it works:\n1.  **Continuous Calling:** Inside the `while True` loop, the `openai_client.responses.create` method is called, sending the current message history to the model and getting a response.\n2.  **Checking for Function Calls:** After receiving the model\'s response, the code iterates through its output to check if any items are of type `"function_call"`.\n3.  **Flag for Continuation:** A boolean flag, `has_function_calls`, is set to `True` if any function calls are detected in the model\'s output. If no function calls are found, this flag remains `False`.\n4.  **Loop Termination:** At the end of each iteration, the loop checks the `has_function_calls` flag. If `has_function_calls` is `False` (meaning the model returned a response without any function calls), the `break` statement is executed, and the loop stops.\n\nIn essence, the loop continues to call the model as long as the model requests to

Token without chunking:   token_count=7963
Token with chunk : token_count=2615

**Answer: 3x fewer**

## Q6. Turning it into an agent
So far search runs once, with the exact query. Let's make it agentic: give the LLM a search tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:


uv add toyaikit
Create a search function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your search tool and run it (with toyaikit, the same way as in the ToyAIKit lesson). Use these instructions for the agent (they nudge it to search a few times):

You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

Ask it:

- How does the agentic loop work, and how is it different from plain RAG?

- The agent decides on its own when to search and when to answer. Count how many times it called the search tool.

- How many times did the agent call search?

Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI gpt-5.4-mini; with a different model or provider the number may differ, so keep that in mind.

0

4

10

20

However, you don't even need ToyAIKit, LangChain, or PydanticAI to replicate this. The official Google GenAI SDK has an identical framework built-in via .chats.create(). It handles the automatic function execution, looping, and history memory for you.

Here is how you replace the entire ToyAIKit script to answer Q6 directly using the Google SDK:

**1. Create the Tool Google**

GenAI lets you pass the raw Python function directly as a tool. All you need is the docstring and the type hint!

In [17]:
def search(query: str) -> dict:
    """
    Search the knowledge base for entries matching the given query.
    """
    # Assuming you already chunked and indexed your documents earlier in the homework
    results = index.search(query, num_results=5)
    return {"results": results}

**2. The Chat Interface and Runner**

Instead of importing runners and callbacks from ToyAIKit, we just start a chat session and pass the search function into the tools array.

In [18]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."
# This `chat` object completely replaces the OpenAIResponsesRunner from ToyAIKit!
chat = genai_client.chats.create(
    model='gemini-2.5-flash',
    config={
        'tools': [search],
        'system_instruction': instructions,
        'temperature': 0.0
    }
)

**3. Running the Prompt**

When you call send_message, the SDK automatically runs the while True loop under the hood. It sees the model wants to search, executes your local Python search function, and loops back to the model until it gets the final answer

In [19]:
response = chat.send_message("How does the agentic loop work, and how is it different from plain RAG?")
print(response.text)

The agentic loop and plain Retrieval Augmented Generation (RAG) are both techniques that enhance Large Language Models (LLMs), but they differ significantly in their approach to problem-solving and interaction with external information.

### Agentic Loop

The agentic loop is an iterative process where an LLM acts as a central controller, dynamically deciding on a sequence of actions to achieve a goal. It works as follows:

1.  **LLM Call:** The LLM receives a prompt or a previous message history.
2.  **Decision and Tool Use:** Based on the input, the LLM decides the next step. This might involve generating a direct answer or, more commonly, calling an external tool (e.g., a search engine, a calculator, a code interpreter) to gather more information or perform a specific task.
3.  **Tool Execution:** If a tool call is made, the tool is executed, and its results are obtained.
4.  **Feedback and Iteration:** The results from the tool execution are then fed back to the LLM as part of the o

**4. Inspect the History to Count the Searches**

To see exactly what the model did (and count the 4 searches), we can loop over the chat history:

In [20]:
search_calls = 0

for message in chat.get_history():
    if message.role == 'model':
        for part in message.parts:
            # Check if the model decided to execute our search tool
            if part.function_call and part.function_call.name == 'search':
                print(f"Agent searched for: {part.function_call.args['query']}")
                search_calls += 1

print(f"Total search calls: {search_calls}")

Agent searched for: agentic loop
Agent searched for: Retrieval Augmented Generation
Total search calls: 2
